In [3]:

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\chity\AppData\Local\Temp\ipykernel_12960\835167180.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\chity\Documents\Projects\Agentic\Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Read all pdf from dir
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 15Projects.pdf
  ✓ Loaded 55 pages

Processing: AI_Guide'26.pdf
  ✓ Loaded 26 pages

Processing: srakshinchityala.pdf
  ✓ Loaded 1 pages

Total documents loaded: 82


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 0, 'page_label': '1', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='15  AI  Engineering  Projects \nThat  Actually  Land  Jobs  \nA  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers \nTransitioning  into  AI  Engineering  \nBASWE   |   AiEngineerAccelerator™'),
 Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 1, 'page_label': '2', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='How  to  Use  This  Guide  \nThis  guide  contains  fifteen  complete  project  build  plans.  Each  one  is  designed  to  take  a  \nmi

In [6]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 82 documents into 246 chunks

Example chunk:
Content: 15  AI  Engineering  Projects 
That  Actually  Land  Jobs  
A  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers 
Transitioning  into  AI  Engineering  
BASWE   |   AiEngineerAccelerator...
Metadata: {'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 0, 'page_label': '1', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 0, 'page_label': '1', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='15  AI  Engineering  Projects \nThat  Actually  Land  Jobs  \nA  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers \nTransitioning  into  AI  Engineering  \nBASWE   |   AiEngineerAccelerator™'),
 Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 1, 'page_label': '2', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='How  to  Use  This  Guide  \nThis  guide  contains  fifteen  complete  project  build  plans.  Each  one  is  designed  to  take  a  \nmi

In [8]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings 
    
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2168.14it/s]


Model loaded successfully. Embedding dimension: 384


In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 246


In [11]:

chunks

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 0, 'page_label': '1', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='15  AI  Engineering  Projects \nThat  Actually  Land  Jobs  \nA  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers \nTransitioning  into  AI  Engineering  \nBASWE   |   AiEngineerAccelerator™'),
 Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx', 'source': '..\\data\\pdf\\15Projects.pdf', 'total_pages': 55, 'page': 1, 'page_label': '2', 'source_file': '15Projects.pdf', 'file_type': 'pdf'}, page_content='How  to  Use  This  Guide  \nThis  guide  contains  fifteen  complete  project  build  plans.  Each  one  is  designed  to  take  a  \nmi

In [12]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 246 texts...


Batches: 100%|██████████| 8/8 [00:07<00:00,  1.08it/s]


Generated embeddings with shape: (246, 384)
Adding 246 documents to vector store...
Successfully added 246 documents to vector store
Total documents in collection: 492


In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What are 15 AI Engineering Projects?")

Retrieving documents for query: 'What are 15 AI Engineering Projects?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e0a41093_0',
  'content': '15  AI  Engineering  Projects \nThat  Actually  Land  Jobs  \nA  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers \nTransitioning  into  AI  Engineering  \nBASWE   |   AiEngineerAccelerator™',
  'metadata': {'page': 0,
   'total_pages': 55,
   'source': '..\\data\\pdf\\15Projects.pdf',
   'creationdate': '',
   'file_type': 'pdf',
   'page_label': '1',
   'title': 'BASWE__15_AI_Engineering_Projects_Guide.docx',
   'producer': 'Skia/PDF m151 Google Docs Renderer',
   'content_length': 201,
   'creator': 'PyPDF',
   'doc_index': 0,
   'source_file': '15Projects.pdf'},
  'similarity_score': 0.5529493987560272,
  'distance': 0.4470506012439728,
  'rank': 1},
 {'id': 'doc_43c248e9_0',
  'content': '15  AI  Engineering  Projects \nThat  Actually  Land  Jobs  \nA  Step-by-Step  Build  Guide  for  Mid-Level  Software  Engineers \nTransitioning  into  AI  Engineering  \nBASWE   |   AiEngineerAccelerator™',
  'metadata': {'content_length': 2

In [17]:
rag_retriever.retrieve("The new graduate ai engineering fast track")

Retrieving documents for query: 'The new graduate ai engineering fast track'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e6e8a09b_210',
  'content': 'baswe.Ai Engineer Accelerator™\nPage 11\n CHAPTER 02\n THE NEW GRAD AI ENGINEERING\n FAST TRACK\n For Students & Recent Graduates Breaking Into AI\n You don’t have years of experience yet. That’s not the problem you think it\n is.',
  'metadata': {'content_length': 223,
   'source_file': "AI_Guide'26.pdf",
   'page_label': '11',
   'file_type': 'pdf',
   'source': "..\\data\\pdf\\AI_Guide'26.pdf",
   'producer': 'ReportLab PDF Library - (opensource)',
   'creationdate': '2026-06-04T17:03:41+00:00',
   'creator': '(unspecified)',
   'keywords': '',
   'moddate': '2026-06-04T17:03:41+00:00',
   'subject': '(unspecified)',
   'author': '(anonymous)',
   'total_pages': 26,
   'title': '(anonymous)',
   'page': 10,
   'doc_index': 210,
   'trapped': '/False'},
  'similarity_score': 0.5009737610816956,
  'distance': 0.49902623891830444,
  'rank': 1},
 {'id': 'doc_f48e5b0e_210',
  'content': 'baswe.Ai Engineer Accelerator™\nPage 11\n CHAPTER 02\n THE 